# M2LLM-Driven Centralized DDPG — Crop Disease UAV
Comparative baseline adapted from: *Trajectory Design and Beamforming with M2LLM-Driven DRL*.

**Architecture:**
- `LLMEncoder`: deep MLP with LayerNorm + GELU — simulates LLaVA hidden-layer projection.
- `CentralDDPGActor`: single agent controlling all 4 UAVs jointly (12-dim action).
- `CentralDDPGCritic`: Q(joint_state, joint_action) with target networks + soft update.

**Environment extension (`uav_env_beamforming.py`):**
- Each UAV action: `[vx, vy, intensity]` — intensity maps beamforming resource allocation.
- Higher intensity → faster treatment (fewer days), plus `tx_quality_bonus` in reward.
- All disease dynamics identical to `uav_env_5`.

In [ ]:
import os, sys
ON_KAGGLE = os.path.exists('/kaggle/input')
if ON_KAGGLE:
    MODULE_DIR = '/kaggle/input/uav-fix5'
else:
    # Notebook lives in new-implementation/; env + networks files are in the same dir
    MODULE_DIR = os.getcwd()
sys.path.insert(0, MODULE_DIR)
print('Module dir:', MODULE_DIR)

In [ ]:
import math, random, time, collections
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
import matplotlib.pyplot as plt

from uav_env_beamforming import (
    UAVFieldEnvBeamforming,
    N_UAVS, N_SECTORS, OBS_SIZE, JOINT_SIZE,
    ACTION_DIM, JOINT_ACTION_DIM,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE, '| JOINT_SIZE:', JOINT_SIZE, '| JOINT_ACTION_DIM:', JOINT_ACTION_DIM)

## Network Architectures
Imported from `networks_beamforming.py`.
- `LLMEncoder`: 512→512→256 MLP with LayerNorm + GELU — simulates LLaVA hidden-layer projection.
- `CentralDDPGActor`: joint_obs (880) → LLMEncoder → 128 → JOINT_ACTION_DIM (12) → Tanh.
- `CentralDDPGCritic`: obs_emb (256) + joint_action (12) → 256 → 128 → Q scalar.

In [ ]:
from networks_beamforming import LLMEncoder, CentralDDPGActor, CentralDDPGCritic

actor         = CentralDDPGActor().to(DEVICE)
critic        = CentralDDPGCritic().to(DEVICE)
target_actor  = CentralDDPGActor().to(DEVICE)
target_critic = CentralDDPGCritic().to(DEVICE)
target_actor.load_state_dict(actor.state_dict())
target_critic.load_state_dict(critic.state_dict())

actor_opt  = Adam(actor.parameters(),  lr=1e-4)
critic_opt = Adam(critic.parameters(), lr=1e-3)
print('Actor  params:', sum(p.numel() for p in actor.parameters()))
print('Critic params:', sum(p.numel() for p in critic.parameters()))

## Replay Buffer & OUNoise

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity: int = 200_000):
        self.buf = collections.deque(maxlen=capacity)

    def push(self, s, a, r, s2, done):
        self.buf.append((s, a, r, s2, done))

    def sample(self, batch_size: int):
        batch = random.sample(self.buf, batch_size)
        s, a, r, s2, d = map(np.array, zip(*batch))
        return (
            torch.tensor(s,  dtype=torch.float32, device=DEVICE),
            torch.tensor(a,  dtype=torch.float32, device=DEVICE),
            torch.tensor(r,  dtype=torch.float32, device=DEVICE).unsqueeze(1),
            torch.tensor(s2, dtype=torch.float32, device=DEVICE),
            torch.tensor(d,  dtype=torch.float32, device=DEVICE).unsqueeze(1),
        )

    def __len__(self):
        return len(self.buf)


class OUNoise:
    """Ornstein-Uhlenbeck noise for DDPG exploration."""
    def __init__(self, dim: int, mu: float = 0.0, theta: float = 0.15,
                 sigma: float = 0.2):
        self.dim   = dim
        self.mu    = mu
        self.theta = theta
        self.sigma = sigma
        self.state = np.zeros(dim)

    def reset(self):
        self.state = np.zeros(self.dim)

    def sample(self):
        dx = self.theta * (self.mu - self.state) + self.sigma * np.random.randn(self.dim)
        self.state += dx
        return self.state.copy()

## Hyperparameters

In [ ]:
EPISODES      = 400
GAMMA_DDPG    = 0.99
TAU_SOFT      = 0.005   # soft target update rate
BATCH_SIZE    = 256
WARMUP_STEPS  = 2000    # random actions before training
REWARD_CLIP   = 50.0
NOISE_DECAY   = 0.999   # multiply sigma each episode
NOISE_MIN     = 0.05

replay = ReplayBuffer()
noise  = OUNoise(JOINT_ACTION_DIM)
global_step = 0

## DDPG Update

In [ ]:
def ddpg_update():
    if len(replay) < BATCH_SIZE:
        return None, None
    s, a, r, s2, done = replay.sample(BATCH_SIZE)

    # Critic update
    with torch.no_grad():
        a2     = target_actor(s2)
        q_next = target_critic(s2, a2)
        y      = torch.clamp(r, -REWARD_CLIP, REWARD_CLIP) + GAMMA_DDPG * q_next * (1 - done)
    q_pred  = critic(s, a)
    c_loss  = F.mse_loss(q_pred, y)
    critic_opt.zero_grad()
    c_loss.backward()
    nn.utils.clip_grad_norm_(critic.parameters(), 1.0)
    critic_opt.step()

    # Actor update
    a_pred = actor(s)
    a_loss = -critic(s, a_pred).mean()
    actor_opt.zero_grad()
    a_loss.backward()
    nn.utils.clip_grad_norm_(actor.parameters(), 1.0)
    actor_opt.step()

    # Soft target updates
    for tp, p in zip(target_actor.parameters(),  actor.parameters()):
        tp.data.mul_(1 - TAU_SOFT).add_(TAU_SOFT * p.data)
    for tp, p in zip(target_critic.parameters(), critic.parameters()):
        tp.data.mul_(1 - TAU_SOFT).add_(TAU_SOFT * p.data)

    return c_loss.item(), a_loss.item()

## Training Loop

In [ ]:
env = UAVFieldEnvBeamforming(seed=42)

ep_rewards   = []
ep_infected  = []
ep_diagnosed = []
ep_crashes   = []
ep_c_losses  = []
ep_a_losses  = []

t0 = time.time()
for ep in range(EPISODES):
    obs_list   = env.reset()
    joint_obs  = np.concatenate(obs_list).astype(np.float32)
    noise.reset()
    noise.sigma = max(noise.sigma * NOISE_DECAY, NOISE_MIN)

    ep_r = 0.0
    crashes_ep = 0
    c_loss_ep, a_loss_ep = [], []

    for _ in range(env.total_steps):
        # Select action
        if global_step < WARMUP_STEPS:
            joint_action = np.random.uniform(-1, 1, JOINT_ACTION_DIM).astype(np.float32)
        else:
            with torch.no_grad():
                obs_t = torch.tensor(joint_obs, dtype=torch.float32,
                                     device=DEVICE).unsqueeze(0)
                joint_action = actor(obs_t).squeeze(0).cpu().numpy()
            joint_action = np.clip(joint_action + noise.sample(), -1.0, 1.0)

        next_obs_list, rewards, done, info = env.step(joint_action)
        next_joint_obs = np.concatenate(next_obs_list).astype(np.float32)

        total_reward = float(sum(rewards))
        replay.push(joint_obs, joint_action, total_reward, next_joint_obs, float(done))
        global_step += 1

        cl, al = ddpg_update()
        if cl is not None:
            c_loss_ep.append(cl)
            a_loss_ep.append(al)

        ep_r += total_reward
        crashes_ep += sum(info['newly_crashed'])
        joint_obs = next_joint_obs
        if done:
            break

    ep_rewards.append(ep_r)
    ep_infected.append(float(env.true_status.sum()) / N_SECTORS)
    ep_diagnosed.append(float(env.ever_diagnosed.sum()) / N_SECTORS)
    ep_crashes.append(crashes_ep)
    ep_c_losses.append(np.mean(c_loss_ep) if c_loss_ep else float('nan'))
    ep_a_losses.append(np.mean(a_loss_ep) if a_loss_ep else float('nan'))

    if (ep + 1) % 50 == 0:
        elapsed = time.time() - t0
        print(f'Ep {ep+1:4d}/{EPISODES} | reward={ep_r:8.1f} '
              f'| inf={ep_infected[-1]:.2f} diag={ep_diagnosed[-1]:.2f} '
              f'| crashes={crashes_ep} | noise_sigma={noise.sigma:.3f} '
              f'| {elapsed:.0f}s')

## Metrics & Plots

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
fig.suptitle('M2LLM-Driven Centralized DDPG — Training Curves', fontsize=14)

def smooth(x, w=20):
    return np.convolve(x, np.ones(w)/w, mode='valid')

axes[0,0].plot(smooth(ep_rewards))
axes[0,0].set_title('Episode Reward')

axes[0,1].plot(smooth(ep_infected), color='red')
axes[0,1].axhline(0.35, color='gray', linestyle='--', label='endemic baseline')
axes[0,1].set_title('Disease Fraction at End')
axes[0,1].legend()

axes[0,2].plot(smooth(ep_diagnosed), color='green')
axes[0,2].set_title('Diagnosed Fraction')

axes[1,0].plot(smooth(ep_crashes), color='orange')
axes[1,0].set_title('Crashes per Episode')

valid_c = [x for x in ep_c_losses if not math.isnan(x)]
axes[1,1].plot(smooth(valid_c), color='purple')
axes[1,1].set_title('Critic Loss')

valid_a = [x for x in ep_a_losses if not math.isnan(x)]
axes[1,2].plot(smooth(valid_a), color='teal')
axes[1,2].set_title('Actor Loss')

plt.tight_layout()
plt.savefig('m2llm_ddpg_training.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved m2llm_ddpg_training.png')

In [ ]:
last = slice(-50, None)
print('=== M2LLM DDPG — Final 50 Episodes ===')
print(f'Mean reward       : {np.mean(ep_rewards[last]):.2f}')
print(f'Mean disease frac : {np.mean(ep_infected[last]):.3f}')
print(f'Mean diag frac    : {np.mean(ep_diagnosed[last]):.3f}')
print(f'Mean crashes/ep   : {np.mean(ep_crashes[last]):.2f}')

In [ ]:
SAVE_DIR = '/kaggle/working' if ON_KAGGLE else '.'
torch.save(actor.state_dict(),         os.path.join(SAVE_DIR, 'm2llm_actor.pt'))
torch.save(critic.state_dict(),        os.path.join(SAVE_DIR, 'm2llm_critic.pt'))
torch.save(target_actor.state_dict(),  os.path.join(SAVE_DIR, 'm2llm_target_actor.pt'))
torch.save(target_critic.state_dict(), os.path.join(SAVE_DIR, 'm2llm_target_critic.pt'))
print('Saved m2llm actor/critic weights to', SAVE_DIR)